In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_csv(r"C:\Users\HP\Documents\GitHub\data-viz-class-material\data\netflix_catalogue.csv")
print(f"Loaded: {len(df)} titles")
print(df['type'].value_counts())
print(df.head())

In [ ]:
print("Genres:", df['genre'].value_counts().head(8))
print("\nCountries:", df['country'].value_counts().head(8))
print("\nRatings:", df['rating'].value_counts())

In [ ]:


# Task 1 — Heatmap: Netflix content by rating and release decade.

import pandas as pd
import plotly.express as px

# Load dataset
df = pd.read_csv(r"C:\Users\HP\Documents\GitHub\data-viz-class-material\data\netflix_catalogue.csv")

# Create decade column
df['decade'] = (df['release_year'] // 10 * 10).astype(str) + 's'

# Filter selected ratings
ratings_filter = ['TV-14', 'TV-MA', 'PG-13', 'R', 'PG']
filtered_df = df[df['rating'].isin(ratings_filter)]

# Create pivot table for heatmap
heatmap_data = (
    filtered_df
    .groupby(['rating', 'decade'])
    .size()
    .reset_index(name='count')
)

pivot_table = heatmap_data.pivot(
    index='rating',
    columns='decade',
    values='count'
).fillna(0)

# Order ratings
rating_order = ['TV-MA', 'TV-14', 'R', 'PG-13', 'PG']
pivot_table = pivot_table.reindex(rating_order)

# Create heatmap
fig = px.imshow(
    pivot_table,
    text_auto=True,
    color_continuous_scale=['#05668d', '#02c39a', '#f0f3bd'],
    aspect='auto',
    labels=dict(
        x="Release Decade",
        y="Content Rating",
        color="Number of Titles"
    ),
    title="TV-MA Dominates Netflix Content in the 2010s and 2020s"
)

# Clean layout
fig.update_layout(
    title_x=0.5,
    font=dict(size=14),
    width=900,
    height=500
)

fig.show()



In [ ]:
# =========================================================
# TASK 2 — WATERFALL: MOVIE ADDITIONS BY YEAR
# =========================================================

# Filter Movies only
movies_df = df[df['type'] == 'Movie']

# Filter years 2015–2022
movies_df = movies_df[
    (movies_df['added_year'] >= 2015) &
    (movies_df['added_year'] <= 2022)
]

# Count movie additions per year
yearly_additions = (
    movies_df
    .groupby('added_year')
    .size()
    .reset_index(name='count')
)

# Find year with largest additions
max_row = yearly_additions.loc[yearly_additions['count'].idxmax()]

max_year = int(max_row['added_year'])
max_count = int(max_row['count'])

# Create waterfall chart
fig2 = go.Figure(go.Waterfall(

    name="Movie Growth",

    orientation="v",

    measure=["relative"] * len(yearly_additions) + ["total"],

    x=[str(year) for year in yearly_additions['added_year']] + ["Total"],

    y=yearly_additions['count'].tolist() + [0],

    increasing={
        "marker": {"color": "#52b788"}   # aesthetic green
    },

    totals={
        "marker": {"color": "#4361ee"}   # aesthetic blue
    },

    connector={
        "line": {"color": "rgba(120,120,120,0.4)"}
    }
))

# Annotation for largest growth
fig2.add_annotation(
    x=str(max_year),
    y=max_count,
    text=f"Peak growth: {max_count} movies",
    showarrow=True,
    arrowhead=2
)

# Layout
fig2.update_layout(
    title="Netflix's Movie Catalogue Expanded Rapidly After 2015",
    title_x=0.5,
    xaxis_title="Year Added",
    yaxis_title="Movies Added",
    width=950,
    height=550,
    font=dict(size=14)
)

fig2.show()